In [1]:
%pip install minigrid

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: minigrid in c:\users\marko\appdata\local\programs\python\python310\lib\site-packages (3.1.0)



In [1]:
import gymnasium as gym
import minigrid
from minigrid.core.constants import OBJECT_TO_IDX
import numpy as np

from utils.minigrid_mapper import PersistentMiniGridMapper
 
env = gym.make(
    "MiniGrid-Empty-5x5-v0",
    render_mode="human"
)

obs, info = env.reset()

print("Action space:")
print(env.action_space)

print("\nObservation space:")
print(env.observation_space)

print("\nObservation keys:")
print(obs.keys())

print("\nImage shape:")
print(obs["image"].shape)

print("\nDirection:")
print(obs["direction"])

print("\nMission:")
print(obs["mission"])

Action space:
Discrete(7)

Observation space:
Dict('direction': Discrete(4), 'image': Box(0, 255, (7, 7, 3), uint8), 'mission': MissionSpace(<function EmptyEnv._gen_mission at 0x000001AE98E37C70>, None))

Observation keys:
dict_keys(['image', 'direction', 'mission'])

Image shape:
(7, 7, 3)

Direction:
0

Mission:
get to the green goal square


c:\Users\Marko\AppData\Local\Programs\Python\Python310\lib\site-packages\pygame\sysfont.py:500: UserWarning: The system font 'freesansbold.ttf' couldn't be found. Did you mean: 'freesansbold', 'freesans'? Verify your font name input. Using the default font instead.
  warnings.warn(


In [2]:
obs, info = env.reset()

terminated = False
truncated = False

while not (terminated or truncated):

    action = env.action_space.sample()

    obs, reward, terminated, truncated, info = env.step(action)

    print(
        f"Action: {action}, "
        f"Reward: {reward:.3f}"
    )

env.close()

Action: 3, Reward: 0.000
Action: 3, Reward: 0.000
Action: 4, Reward: 0.000
Action: 5, Reward: 0.000
Action: 6, Reward: 0.000
Action: 2, Reward: 0.000
Action: 5, Reward: 0.000
Action: 6, Reward: 0.000
Action: 1, Reward: 0.000
Action: 1, Reward: 0.000
Action: 3, Reward: 0.000
Action: 1, Reward: 0.000
Action: 2, Reward: 0.000
Action: 4, Reward: 0.000
Action: 1, Reward: 0.000
Action: 0, Reward: 0.000
Action: 5, Reward: 0.000
Action: 3, Reward: 0.000
Action: 3, Reward: 0.000
Action: 1, Reward: 0.000
Action: 1, Reward: 0.000
Action: 2, Reward: 0.000
Action: 4, Reward: 0.000
Action: 2, Reward: 0.000
Action: 0, Reward: 0.000
Action: 5, Reward: 0.000
Action: 2, Reward: 0.757


## Debug-only mapper validation against MiniGrid ground truth

This test uses MiniGrid's privileged internal map and agent pose only to validate `PersistentMiniGridMapper`; none of this information is supplied to a policy or learning algorithm. The mapper chooses `(0, 0)` as the episode's starting coordinate, so a mapped coordinate must equal the corresponding true coordinate minus the true starting position. `MiniGrid-DoorKey-8x8-v0` provides randomized layouts, objects, and door states that exercise more mapping behavior than the basic empty environment.

In [3]:
# This validation is intentionally separate from all agent inputs.
MAPPER_VALIDATION_EPISODES = 100
MAPPER_VALIDATION_SEED = 20_000
MAPPER_TEST_ENVIRONMENT = "MiniGrid-DoorKey-8x8-v0"


def true_cell_encoding(grid, position):
    """Return a MiniGrid-compatible encoding for one true grid cell."""
    x, y = map(int, position)
    cell = grid.get(x, y)

    if cell is None:
        # MiniGrid represents an empty grid location as None internally.
        return (OBJECT_TO_IDX["empty"], 0, 0)

    return tuple(map(int, cell.encode()))


def validate_mapper_state(mapper, unwrapped_env, true_start_position):
    """Assert that pose and all known cells agree with ground truth."""
    true_position = np.asarray(unwrapped_env.agent_pos, dtype=int)
    expected_mapped_position = true_position - true_start_position

    np.testing.assert_array_equal(
        mapper.position,
        expected_mapped_position,
        err_msg="Mapped agent position disagrees with MiniGrid",
    )
    assert mapper.direction == int(unwrapped_env.agent_dir), (
        "Mapped direction disagrees with MiniGrid: "
        f"mapped={mapper.direction}, true={unwrapped_env.agent_dir}"
    )

    checked_cells = 0

    for mapped_position, mapped_cell in mapper.cells.items():
        true_grid_position = (
            np.asarray(mapped_position, dtype=int)
            + true_start_position
        )
        expected_encoding = true_cell_encoding(
            unwrapped_env.grid, true_grid_position
        )
        mapped_encoding = (
            mapped_cell.object_idx,
            mapped_cell.color_idx,
            mapped_cell.state,
        )

        assert mapped_encoding == expected_encoding, (
            f"Cell mismatch at mapped coordinate {mapped_position} "
            f"(true coordinate {tuple(true_grid_position)}): "
            f"mapped={mapped_encoding}, true={expected_encoding}"
        )
        checked_cells += 1

    return checked_cells


validation_env = gym.make(MAPPER_TEST_ENVIRONMENT)
observations_checked = 0
mapped_cells_checked = 0

try:
    for validation_episode in range(MAPPER_VALIDATION_EPISODES):
        episode_seed = MAPPER_VALIDATION_SEED + validation_episode
        validation_env.action_space.seed(episode_seed)
        observation, info = validation_env.reset(seed=episode_seed)

        unwrapped_env = validation_env.unwrapped
        true_start_position = np.asarray(
            unwrapped_env.agent_pos, dtype=int
        ).copy()
        mapper = PersistentMiniGridMapper()
        mapper.reset(observation)

        mapped_cells_checked += validate_mapper_state(
            mapper, unwrapped_env, true_start_position
        )
        observations_checked += 1

        terminated = False
        truncated = False

        while not (terminated or truncated):
            action = validation_env.action_space.sample()

            # Predict only from the chosen action and accumulated map.
            mapper.predict_action(action)
            observation, reward, terminated, truncated, info = (
                validation_env.step(action)
            )
            mapper.observe(observation)

            mapped_cells_checked += validate_mapper_state(
                mapper, unwrapped_env, true_start_position
            )
            observations_checked += 1
finally:
    validation_env.close()

print("Mapper validation passed with zero inconsistencies.")
print("Random episodes:", MAPPER_VALIDATION_EPISODES)
print("Observations checked:", observations_checked)
print("Mapped cell comparisons:", mapped_cells_checked)

Mapper validation passed with zero inconsistencies.
Random episodes: 100
Observations checked: 63450
Mapped cell comparisons: 2325413
